# Stage 4b — Post-Response Extraction (no-crop, alignment-safe)

Rewritten after Stage 4 hit a wall: the expected `excludedchannels` field does not
exist in this dataset release. This version **never crops and never guesses**.

Strategy:
1. Try to recover electrode identity from `crr_channels` (the field not yet checked).
2. Whatever happens, store each subject's data in a 64-row array, writing into the
   rows we can justify and leaving the rest as `NaN`. Nothing is silently shifted.
3. Record a per-epoch flag `aligned` saying whether that epoch's channel identity is
   trustworthy.

Even if identity recovery fails entirely, the **15 subjects with the complete
64-channel montage** are aligned by definition (no electrodes removed, so row *i*
is electrode *i*). Those alone give 181 overconfident vs 446 appropriately-uncertain
trials across all 15 subjects — enough for a properly powered pooled RQ3 with
leave-one-subject-out CV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get update -qq && apt-get install -y octave -qq
print("ready")

In [ ]:
DATA_DIR = "/content/drive/MyDrive/PATH_TO_YOUR_EEG_FOLDER"   # <-- EDIT
OUT_PATH = "/content/drive/MyDrive/eeg_epochs_postresp.npz"

PRE_MS, POST_MS = 100, 600      # response-locked: covers ERN (0-100) and Pe (200-400)
ORIG_FS, TARGET_FS = 1000, 250
MONTAGE = 64

DS = ORIG_FS // TARGET_FS
TOTAL_MS = PRE_MS + POST_MS
N_OUT = int(TOTAL_MS/1000*TARGET_FS)
print(f"window -{PRE_MS} to +{POST_MS} ms | {N_OUT} samples/trial @ {TARGET_FS} Hz")

In [ ]:
import os, glob, re, subprocess, gc, time
import numpy as np
from scipy.io import loadmat

pat = re.compile(r"sub-(\d+)_run-(\d+)")
def key_of(p):
    m = pat.search(os.path.basename(p)); return (m.group(1), m.group(2)) if m else None
dmap = {key_of(f): f for f in glob.glob(os.path.join(DATA_DIR,"**","EEG_data_sub-*.mat"), recursive=True) if key_of(f)}
emap = {key_of(f): f for f in glob.glob(os.path.join(DATA_DIR,"**","EEG_events_sub-*.mat"), recursive=True) if key_of(f)}
paired = sorted(set(dmap) & set(emap))
print(f"{len(paired)} subject/run pairs")

In [ ]:
# Loader: pull Y plus every candidate identity field, without assuming any exist.
def load_run(mat_path, tmp="/content/_t.mat"):
    if os.path.exists(tmp): os.remove(tmp)
    cmd = (f"load('{mat_path}','EEGdata'); Y=EEGdata.Y; "
           f"crr = []; chn = []; nix = []; "
           f"if isfield(EEGdata,'crr_channels'); crr=EEGdata.crr_channels; end; "
           f"if isfield(EEGdata,'chnan'); chn=EEGdata.chnan; end; "
           f"if isfield(EEGdata,'nanidx'); nix=EEGdata.nanidx; end; "
           f"save('-mat7-binary','{tmp}','Y','crr','chn','nix');")
    r = subprocess.run(["octave","--no-gui","--quiet","--eval",cmd],
                       capture_output=True, text=True)
    if not os.path.exists(tmp):
        raise RuntimeError(f"Octave failed: {r.stderr[:400]}")
    m = loadmat(tmp)
    def vec(k):
        a = np.array(m.get(k, [])).squeeze()
        a = np.atleast_1d(a)
        return a if a.size and a.dtype.kind in "iuf" else np.array([])
    out = (m["Y"], vec("crr"), vec("chn"), vec("nix"))
    os.remove(tmp); return out


def resolve_ids(n_ch, crr, chn, nix):
    """Return (electrode_ids array of length n_ch, aligned_flag, reason).
    Only returns aligned=True when the mapping is actually justified."""
    # Case A: full montage, nothing removed -> identity is trivially known
    if n_ch == MONTAGE:
        return np.arange(1, MONTAGE+1), True, "full montage"
    # Case B: crr_channels lists the RETAINED electrodes
    for cand, nm in [(crr,"crr_channels"), (chn,"chnan"), (nix,"nanidx")]:
        c = np.asarray(cand).astype(int).ravel()
        if c.size == n_ch and c.min() >= 1 and c.max() <= MONTAGE and len(set(c.tolist())) == n_ch:
            return c, True, f"{nm} = retained list"
    # Case C: field lists the EXCLUDED electrodes
    for cand, nm in [(crr,"crr_channels"), (chn,"chnan"), (nix,"nanidx")]:
        c = np.asarray(cand).astype(int).ravel()
        if 0 < c.size == (MONTAGE - n_ch) and c.min() >= 1 and c.max() <= MONTAGE:
            keep = np.array([e for e in range(1, MONTAGE+1) if e not in set(c.tolist())])
            if len(keep) == n_ch:
                return keep, True, f"{nm} = excluded list"
    # Case D: cannot justify a mapping
    return np.arange(1, n_ch+1), False, "UNRESOLVED"

In [ ]:
# Probe ONE file with excluded channels before the long run
probe_key = next(k for k in paired if k[0] in ("01","04","07","10","12","13","21"))
print("probing", dmap[probe_key])
Y, crr, chn, nix = load_run(dmap[probe_key])
print("size(Y) =", Y.shape)
for nm, v in [("crr_channels",crr), ("chnan",chn), ("nanidx",nix)]:
    print(f"  {nm}: size={np.asarray(v).size}  values={np.asarray(v).ravel()[:70]}")
ids, ok, why = resolve_ids(Y.shape[0], crr, chn, nix)
print(f"\nresolution -> aligned={ok}  reason='{why}'")
print("If aligned=False here, the run still proceeds; those subjects are simply")
print("flagged and excluded from pooled analyses (the 64-channel subjects remain usable).")
del Y; gc.collect()

In [ ]:
# MAIN LOOP
X_all=[]; aligned_all=[]; reason_all=[]
subj_all=[]; run_all=[]; conf_all=[]; acc_all=[]; ch_all=[]; dd_all=[]; rt_all=[]; ti_all=[]
log=[]; t0=time.time()

for i,key in enumerate(paired):
    s,r = key
    print(f"[{i+1}/{len(paired)}] sub-{s} run-{r} ...", end=" ", flush=True)
    try:
        ev = loadmat(emap[key])
        tstim=ev["tstim"].squeeze().astype(float); RT=ev["RT"].squeeze().astype(float)
        cf=ev["confidence"].squeeze().astype(int); ac=ev["accuracy"].squeeze().astype(int)
        cho=ev["choice"].squeeze().astype(int); dd=ev["dotdirection"].squeeze().astype(int)

        Y, crr, chn, nix = load_run(dmap[key])
        n_ch, n_samp = Y.shape
        ids, ok, why = resolve_ids(n_ch, crr, chn, nix)

        kept=skipped=0
        for t in range(len(tstim)):
            resp = tstim[t] + RT[t]
            ws = int(round(resp - PRE_MS)); we = ws + TOTAL_MS
            if ws < 0 or we > n_samp: skipped += 1; continue
            seg = Y[:, ws:we]
            if seg.shape[1] != TOTAL_MS: skipped += 1; continue
            seg = seg[:, :DS*N_OUT].reshape(n_ch, N_OUT, DS).mean(axis=2).astype(np.float32)

            # write into a 64-row canvas; unresolved subjects fill the first n_ch rows
            canvas = np.full((MONTAGE, N_OUT), np.nan, dtype=np.float32)
            for row, eid in enumerate(ids):
                if 1 <= eid <= MONTAGE: canvas[eid-1] = seg[row]
            X_all.append(canvas); aligned_all.append(ok); reason_all.append(why)
            subj_all.append(s); run_all.append(r)
            conf_all.append(cf[t]); acc_all.append(ac[t]); ch_all.append(cho[t])
            dd_all.append(dd[t]); rt_all.append(RT[t]); ti_all.append(t)
            kept += 1

        log.append((s,r,len(tstim),kept,skipped,n_ch,ok,why))
        print(f"kept {kept}/{len(tstim)} ch={n_ch} aligned={ok} ({why})")
        del Y; gc.collect()
    except Exception as e:
        print("FAILED:", e); log.append((s,r,-1,0,-1,-1,False,str(e)[:40]))

print(f"\n{len(X_all)} epochs in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Save
X = np.stack(X_all)
print("X:", X.shape, X.dtype, f"{X.nbytes/1e6:.0f} MB in memory")
np.savez_compressed(OUT_PATH, X=X,
    aligned=np.array(aligned_all), align_reason=np.array(reason_all),
    subject=np.array(subj_all), run=np.array(run_all),
    confidence=np.array(conf_all), accuracy=np.array(acc_all),
    choice=np.array(ch_all), dotdirection=np.array(dd_all),
    RT=np.array(rt_all), trial_index=np.array(ti_all),
    pre_ms=PRE_MS, post_ms=POST_MS, fs=TARGET_FS, montage=MONTAGE)
sz = os.path.getsize(OUT_PATH)/1e6
print(f"saved {OUT_PATH} ({sz:.1f} MB)")
if sz > 480: print("WARNING >480MB: set TARGET_FS=200 or POST_MS=500 and re-run")

In [ ]:
# Report -- paste this back
from collections import Counter
subj_a=np.array(subj_all); acc_a=np.array(acc_all); conf_a=np.array(conf_all)
al=np.array(aligned_all)
print("=== PER-RUN ===")
print(f"{'sub':>4} {'run':>4} {'tot':>5} {'kept':>5} {'skip':>5} {'ch':>4} {'algn':>6}  reason")
for row in log: print("{:>4} {:>4} {:>5} {:>5} {:>5} {:>4} {!s:>6}  {}".format(*row))

print(f"\nEpochs: {len(acc_a)} | aligned: {int(al.sum())} | unresolved: {int((~al).sum())}")
print("Alignment reasons:", dict(Counter(reason_all)))
print("Aligned subjects:", sorted(set(subj_a[al])))

lo=np.isin(conf_a,[1,2,3]); hi=np.isin(conf_a,[7,8,9])
for label, mask in [("ALL EPOCHS", np.ones(len(acc_a),bool)), ("ALIGNED ONLY", al)]:
    print(f"\n=== 4-WAY TABLE, {label} ===")
    print(f"{'':>12} {'Low':>8} {'High':>8}")
    for av,an in [(1,'Correct'),(0,'Incorrect')]:
        m=(acc_a==av)&mask
        print(f"{an:>12} {int((m&lo).sum()):>8} {int((m&hi).sum()):>8}")
    oc=(acc_a==0)&hi&mask
    print(f"  overconfident: {int(oc.sum())} trials from {len(set(subj_a[oc]))} subjects")

## What to send back

Output of the **probe cell** (tells us whether `crr_channels` resolved identity),
and the **report cell**. Then upload `eeg_epochs_postresp.npz` and I'll run RQ3.

Expected outcomes:
- `crr_channels` resolves -> aligned epochs from all 24 subjects; RQ3 pooled on ~294 overconfident trials.
- It doesn't -> aligned = the 15 full-montage subjects; RQ3 pooled on ~181 vs ~446 trials.

Either way RQ3 becomes testable, and the `aligned` flag means no misaligned
data can leak into a pooled analysis by accident.